# HCLE Navigator — Pipeline Model
**Human Capital Loss Engine** | Data panel 34 provinsi Indonesia, 2021-2024


## 0. Setup

In [ ]:
!pip install -q statsmodels scikit-learn matplotlib tabulate

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, r2_score, mean_squared_error

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option('display.width', 140)
np.random.seed(42)

FILE = 'DATA_PANEL_HCLI_PROVINSI_2021-2024.csv'
CORE = ['NEET', 'P1', 'P2', 'Internet', 'RLS', 'HLS']
TARGET = 'HCLI'
K = 3
SEED = 42
N_INIT = 25

## 1. Muat data dan praproses

Export Google Sheets memakai koma sebagai pemisah desimal. Kalau langsung dilempar ke
`pd.to_numeric(errors='coerce')`, semua nilai jadi `NaN` lalu `dropna()` mengosongkan
dataframe. Fungsi di bawah menangani kedua format sekaligus.

In [ ]:
def to_num(s):
    """Konversi kolom bertipe teks ke numerik. Menangani desimal koma dan titik ribuan."""
    return pd.to_numeric(
        s.astype(str)
         .str.strip()
         .str.replace('.', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )

df = pd.read_csv(FILE)
df.columns = ['Provinsi', 'Tahun', 'NEET', 'P1', 'P2', 'Internet', 'RLS', 'HLS', 'HCLI']

n_raw = len(df)
for c in CORE + [TARGET]:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = to_num(df[c])

df = df.dropna(subset=CORE + [TARGET]).reset_index(drop=True)

print(f'Baris mentah    : {n_raw}')
print(f'Baris terpakai  : {len(df)}  (dibuang: {n_raw - len(df)})')
print(f'Provinsi        : {df.Provinsi.nunique()}')
print(f'Tahun           : {sorted(df.Tahun.unique())}')
print(f'Panel seimbang  : {len(df) == df.Provinsi.nunique() * df.Tahun.nunique()}')
df[CORE + [TARGET]].describe().T.round(4)

## 2. Audit konstruksi indeks

Langkah ini **wajib** dilakukan sebelum menafsirkan hasil regresi.

HCLI adalah indeks komposit yang disusun dari enam indikator. Kalau indeks itu merupakan
kombinasi linear dari keenamnya, maka meregresikan HCLI terhadap keenam variabel yang sama
bukanlah prediksi, melainkan **rekonstruksi rumus indeksnya sendiri**. R² yang mendekati 1
adalah konsekuensi aritmetika, bukan bukti kemampuan prediktif.

Uji: jalankan pooled OLS tanpa efek tetap, lalu bandingkan simpangan baku residual dengan
noise pembulatan teoretis. HCLI dilaporkan dalam dua desimal, sehingga noise pembulatannya
adalah `0.01 / sqrt(12) = 0.00289`.

In [ ]:
pooled = smf.ols(f'{TARGET} ~ ' + ' + '.join(CORE), data=df).fit()
resid = df[TARGET] - pooled.predict(df)
noise_teoretis = 0.01 / np.sqrt(12)

print(f'Pooled OLS R2 (tanpa efek tetap) : {pooled.rsquared:.6f}')
print(f'SD residual                      : {resid.std():.6f}')
print(f'SD noise pembulatan 2 desimal    : {noise_teoretis:.6f}')
print(f'Rasio                            : {resid.std() / noise_teoretis:.3f}')

cocok = (pooled.predict(df).round(2) == df[TARGET]).sum()
print(f'Rekonstruksi persis (dibulatkan) : {cocok}/{len(df)} baris')
print()
print('Bobot efektif indeks:')
print(pooled.params.round(6).to_string())

if resid.std() / noise_teoretis < 1.5:
    print()
    print('KESIMPULAN: HCLI adalah kombinasi linear dari keenam prediktor.')
    print('Regresi di bawah HARUS ditafsirkan sebagai DEKOMPOSISI BOBOT INDEKS,')
    print('bukan sebagai prediksi atau inferensi kausal.')

## 3. Korelasi dan multikolinearitas

Matriks korelasi menjawab dua hal sekaligus: kekuatan hubungan tiap prediktor dengan
target, dan seberapa besar prediktor saling tumpang tindih. VIF di atas 10 menandakan
multikolinearitas serius.

In [ ]:
corr = df[CORE + [TARGET]].corr()
print('Korelasi Pearson terhadap HCLI:')
print(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3).to_string())
print()

X_vif = sm.add_constant(df[CORE])
vif = pd.DataFrame({
    'Variabel': CORE,
    'VIF': [variance_inflation_factor(X_vif.values, i + 1) for i in range(len(CORE))]
}).sort_values('VIF', ascending=False)
print('Variance Inflation Factor:')
print(vif.round(2).to_string(index=False))
print()
tinggi = vif[vif.VIF > 10].Variabel.tolist()
if tinggi:
    print(f'PERINGATAN: VIF > 10 pada {tinggi}. Koefisien masing-masing tidak stabil.')

In [ ]:
matplotlib.rcParams.update({'font.family': 'serif', 'font.size': 7.5})
labels = CORE + [TARGET]
M = corr.values
mask = np.triu(np.ones_like(M, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(3.35, 2.9), dpi=300)
im = ax.imshow(np.ma.masked_where(mask, M), cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(i + 1):
        ax.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=6,
                color='white' if abs(M[i, j]) > 0.6 else 'black')
fig.colorbar(im, ax=ax, shrink=0.8, pad=0.03)
for s in ax.spines.values(): s.set_visible(False)
fig.tight_layout(pad=0.3)
fig.savefig('fig1_korelasi.png', bbox_inches='tight')
plt.show()

## 4. Layer 1 — Segmentasi wilayah (K-Means)

Klasterisasi memakai potongan lintang 2024 dengan enam prediktor terstandardisasi.
**HCLI tidak dimasukkan sebagai fitur** supaya bisa dipakai sebagai validasi eksternal:
kalau kluster yang terbentuk ternyata punya tingkat HCLI yang berbeda tajam, berarti
segmentasinya menangkap struktur nyata.

In [ ]:
df24 = df[df.Tahun == df.Tahun.max()].copy()
X = df24[CORE].values
Xs = StandardScaler().fit_transform(X)

hasil = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, init='k-means++', random_state=SEED, n_init=N_INIT).fit(Xs)
    hasil.append({'k': k, 'Inertia': km.inertia_, 'Silhouette': silhouette_score(Xs, km.labels_)})
val = pd.DataFrame(hasil)
print(val.round(4).to_string(index=False))
print()
print(f"Silhouette tertinggi pada k = {int(val.loc[val.Silhouette.idxmax(), 'k'])}")
print(f'k yang dipakai         : {K}')

In [ ]:
fig, ax1 = plt.subplots(figsize=(3.35, 2.1), dpi=300)
ax1.plot(val.k, val.Inertia, 'o-', color='#333333', lw=1.2, ms=4)
ax1.set_xlabel('Jumlah kluster (k)'); ax1.set_ylabel('Inertia', color='#333333')
ax2 = ax1.twinx()
ax2.plot(val.k, val.Silhouette, 's--', color='#B03A2E', lw=1.2, ms=4)
ax2.set_ylabel('Silhouette', color='#B03A2E'); ax2.tick_params(axis='y', labelcolor='#B03A2E')
ax1.axvline(K, color='#888888', ls=':', lw=1)
ax1.grid(alpha=0.25, lw=0.5)
fig.tight_layout(pad=0.3)
fig.savefig('fig2_validasi_k.png', bbox_inches='tight')
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=K, init='k-means++', random_state=SEED, n_init=N_INIT)
df24['Kluster'] = ['Kluster ' + str(c + 1) for c in kmeans.fit_predict(Xs)]

profil = df24.groupby('Kluster')[CORE + [TARGET]].mean()
profil.insert(0, 'n', df24.groupby('Kluster').size())
profil = profil.sort_values(TARGET)
print(profil.round(4).to_markdown(floatfmt='.4f'))
print()
print(f'Rasio HCLI kluster terburuk vs terbaik: '
      f'{profil[TARGET].max() / profil[TARGET].min():.2f}x')
print()
for k, g in df24.groupby('Kluster'):
    print(f"{k} ({len(g)}): {', '.join(sorted(g.Provinsi))}")

## 5. Layer 2 — Regresi data panel efek tetap

`C(Provinsi)` memberi intercept tersendiri bagi tiap provinsi, sehingga karakteristik
wilayah yang tidak berubah antarwaktu terserap ke variabel boneka. Koefisien prediktor
dengan demikian dibaca sebagai efek **di dalam provinsi yang sama**, bukan perbandingan
antarprovinsi.

Split train/test bersifat temporal: dilatih 2021-2023, diuji pada tahun terakhir.

In [ ]:
tahun_uji = df.Tahun.max()
train = df[df.Tahun < tahun_uji].copy()
test  = df[df.Tahun == tahun_uji].copy()

formula = f'{TARGET} ~ ' + ' + '.join(CORE) + ' + C(Provinsi)'
fe = smf.ols(formula, data=train).fit()

# clustered standard error per provinsi - wajib untuk data panel
fe_cl = smf.ols(formula, data=train).fit(
    cov_type='cluster', cov_kwds={'groups': train['Provinsi']}
)

print(f'Observasi latih : {len(train)}   uji: {len(test)}')
print(f'Parameter       : {len(fe.params)}')
print(f'Rasio obs/param : {len(train) / len(fe.params):.2f}')

### Koefisien terbaku

Keenam variabel punya rentang yang sangat berbeda (NEET puluhan, P2 di bawah 1,5).
Membandingkan koefisien mentah antarvariabel karena itu tidak setara. Karena modelnya
efek tetap, pembakuan harus memakai **simpangan baku within-province**, yaitu variasi
di dalam provinsi setelah rata-rata provinsinya dibuang. Itulah variasi yang benar-benar
dipakai model.

In [ ]:
dm = df.copy()
for c in CORE + [TARGET]:
    dm[c + '_w'] = dm.groupby('Provinsi')[c].transform(lambda s: s - s.mean())

sd_w = {c: dm[c + '_w'].std(ddof=1) for c in CORE}
sd_target_w = dm[TARGET + '_w'].std(ddof=1)

beta = pd.DataFrame({
    'Variabel': CORE,
    'Beta': [fe.params[c] for c in CORE],
    'p (naif)': [fe.pvalues[c] for c in CORE],
    'p (clustered)': [fe_cl.pvalues[c] for c in CORE],
    'SD within': [sd_w[c] for c in CORE],
})
beta['Beta terbaku'] = beta.Beta * beta['SD within'] / sd_target_w
beta['Efek per 1 SD'] = beta.Beta * beta['SD within']
beta['Sig. clustered'] = beta['p (clustered)'] < 0.05
beta = beta.reindex(beta['Beta terbaku'].abs().sort_values(ascending=False).index)
beta.insert(0, 'Peringkat', range(1, len(beta) + 1))

print(beta.round(5).to_markdown(index=False, floatfmt='.5f'))
print()
print('Peringkat koefisien MENTAH  :',
      ' > '.join(beta.reindex(beta.Beta.abs().sort_values(ascending=False).index).Variabel))
print('Peringkat koefisien TERBAKU :', ' > '.join(beta.Variabel))

## 6. Evaluasi

Perhatikan hasil audit di bagian 2 saat menafsirkan angka di bawah. Kalau HCLI memang
kombinasi linear dari prediktornya, R² yang tinggi **bukan** bukti kemampuan prediktif.
Yang tetap bermakna adalah galatnya: seberapa dekat rekonstruksi indeks pada tahun yang
tidak ikut dilatih.

In [ ]:
test = test.copy()
test['HCLI_prediksi'] = fe.predict(test)

r2  = r2_score(test[TARGET], test.HCLI_prediksi)
mse = mean_squared_error(test[TARGET], test.HCLI_prediksi)

print(f'R2 latih          : {fe.rsquared:.4f}')
print(f'R2 uji {tahun_uji}       : {r2:.4f}')
print(f'MSE uji           : {mse:.7f}')
print(f'RMSE uji          : {np.sqrt(mse):.5f}')
print(f'RMSE / rentang Y  : {np.sqrt(mse) / (df[TARGET].max() - df[TARGET].min()):.2%}')
print()
print('Sebagai pembanding, R2 pooled OLS tanpa efek tetap: '
      f'{pooled.rsquared:.4f} (lihat audit bagian 2)')

In [ ]:
# Tren nasional
tren = df.groupby('Tahun')[[TARGET, 'NEET']].mean().reset_index()
print(tren.round(4).to_markdown(index=False, floatfmt='.4f'))
awal, akhir = tren[TARGET].iloc[0], tren[TARGET].iloc[-1]
print(f'\nPerubahan HCLI: {awal:.4f} -> {akhir:.4f}  ({(akhir - awal) / awal:.1%})')

## 7. Layer 3 — Simulasi skenario

Model dipakai ulang sebagai mesin prediksi kontrafaktual. Fungsi di bawah menerima
perubahan persentase per variabel lalu mengembalikan estimasi HCLI baru beserta
dekomposisi kontribusinya.

In [ ]:
def simulasi(perubahan, kluster=None, basis=None):
    """
    perubahan : dict, misal {'Internet': 0.10, 'NEET': -0.20}
    kluster   : nama kluster untuk membatasi wilayah sasaran, None = nasional
    """
    base = (basis if basis is not None else df24).copy()
    if kluster:
        base = base[base.Kluster == kluster]
    if base.empty:
        raise ValueError('Wilayah sasaran kosong')

    skenario = base.copy()
    for var, delta in perubahan.items():
        skenario[var] = skenario[var] * (1 + delta)

    hcli_awal = fe.predict(base).mean()
    hcli_baru = fe.predict(skenario).mean()

    kontribusi = {
        var: fe.params[var] * base[var].mean() * delta
        for var, delta in perubahan.items()
    }
    return {
        'wilayah': kluster or 'Nasional',
        'n_provinsi': len(base),
        'hcli_awal': hcli_awal,
        'hcli_baru': hcli_baru,
        'delta': hcli_baru - hcli_awal,
        'delta_persen': (hcli_baru - hcli_awal) / hcli_awal,
        'kontribusi': kontribusi,
    }


for nama, skenario in {
    'Internet +10%'          : {'Internet': 0.10},
    'NEET -20%'              : {'NEET': -0.20},
    'P2 -20%'                : {'P2': -0.20},
    'RLS +10%'               : {'RLS': 0.10},
    'Paket gabungan'         : {'Internet': 0.10, 'NEET': -0.10, 'P2': -0.10},
}.items():
    h = simulasi(skenario)
    print(f"{nama:22s} HCLI {h['hcli_awal']:.4f} -> {h['hcli_baru']:.4f}  "
          f"(delta {h['delta']:+.4f}, {h['delta_persen']:+.1%})")

In [ ]:
print('Dampak Internet +10% per kluster:\n')
for k in sorted(df24.Kluster.unique()):
    h = simulasi({'Internet': 0.10}, kluster=k)
    print(f"{k} (n={h['n_provinsi']:2d})  {h['hcli_awal']:.4f} -> "
          f"{h['hcli_baru']:.4f}  ({h['delta_persen']:+.1%})" )

## 8. Uji ketahanan

Dua pemeriksaan: pengaruh multikolinearitas P1-P2, dan stabilitas keanggotaan kluster
kalau benih acaknya diganti.

In [ ]:
fe_tanpa_p1 = smf.ols(f'{TARGET} ~ NEET + P2 + Internet + RLS + HLS + C(Provinsi)',
                      data=train).fit()
print('Model lengkap    : R2 uji = %.4f' % r2)
print('Model tanpa P1   : R2 uji = %.4f'
      % r2_score(test[TARGET], fe_tanpa_p1.predict(test)))
print()
banding = pd.DataFrame({
    'Lengkap': fe.params.filter(items=CORE),
    'Tanpa P1': fe_tanpa_p1.params.filter(items=CORE),
})
print(banding.round(5).to_string())

In [2]:
dasar = KMeans(n_clusters=K, init='k-means++', random_state=SEED, n_init=N_INIT).fit_predict(Xs)
from sklearn.metrics import adjusted_rand_score
skor = [adjusted_rand_score(
dasar,
        KMeans(n_clusters=K, init='k-means++', random_state=s, n_init=N_INIT).fit_predict(Xs))
        for s in range(100, 130)]
print(f'Adjusted Rand Index terhadap 30 benih lain: rata-rata {np.mean(skor):.3f}, '
      f'minimum {np.min(skor):.3f}')
print('(1,000 berarti keanggotaan kluster identik)')

NameError: name 'KMeans' is not defined